# Limpieza de datos — TFG (versión Colab)
**Beatriz Muñoz García-Serrano · Grado en Business Analytics · UFV**

Genera dos datasets limpios a partir de los archivos brutos:
- `encuesta_clean_tfg.csv` — encuesta propia (224 respuestas × 45 variables)
- `infonieve_clean_tfg.csv` — partes de nieve históricos Infonieve (142.893 × 30)

## Archivos que debes tener en `/content/` antes de ejecutar
| Archivo | Descripción |
|---|---|
| `infonieve-partedenieve-historico-estaciones.csv` | Raw Infonieve (10.3 MB, sep=";", latin-1) |
| `Tu experiencia en las estaciones de esquí (respuestas) (3).xlsx` | Encuesta propia (224 filas) |

Los outputs se guardan directamente en `/content/`.


In [ ]:
import os, re, shutil, unicodedata
import numpy as np
import pandas as pd

# ── Rutas Colab ──────────────────────────────────────────────────────────────
CONTENT      = '/content/'
SALIDA_ENC   = CONTENT + 'encuesta_clean_tfg.csv'
SALIDA_INF   = CONTENT + 'infonieve_clean_tfg.csv'

print('✓ Librerías cargadas')
print(f'  Salida encuesta  → {SALIDA_ENC}')
print(f'  Salida infonieve → {SALIDA_INF}')


✓ Librerías cargadas
  Salida encuesta  → /content/encuesta_clean_tfg.csv
  Salida infonieve → /content/infonieve_clean_tfg.csv


## Utilidades comunes

In [ ]:
_RE_EMOJI = re.compile(
    "[\U00010000-\U0010FFFF"
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "\u200d\u2640\u2642\uFE0F"
    "]+",
    flags=re.UNICODE
)

def quitar_emoji(texto):
    if not isinstance(texto, str): return texto
    return _RE_EMOJI.sub('', texto).strip()

def normalizar_clave(texto):
    if not isinstance(texto, str): return ''
    t = texto.strip().lower()
    t = unicodedata.normalize('NFKD', t)
    t = ''.join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r'[^a-z0-9/ ]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def limpiar_texto_libre(texto):
    if pd.isna(texto): return np.nan
    t = quitar_emoji(str(texto)).strip()
    if re.fullmatch(r'[\s.,;:!?¿¡\-–—_/\\|@#*+~`^\'\"(){}\[\]<>]*', t): return np.nan
    return t if t else np.nan

print('✓ Funciones de utilidad definidas')


✓ Funciones de utilidad definidas


## Detección automática de archivos en `/content/`

In [ ]:
def detectar_archivos(directorio):
    archivos = sorted(os.listdir(directorio))
    enc_path = inf_path = None
    for nombre in archivos:
        ruta = os.path.join(directorio, nombre)
        nombre_l = nombre.lower()
        if nombre_l.endswith('.xlsx'):
            if any(kw in nombre_l for kw in ('experiencia', 'respuesta', 'encuesta')):
                enc_path = ruta
            elif enc_path is None:
                try:
                    cab = pd.read_excel(ruta, nrows=0)
                    cabeceras_l = ' '.join(str(c).lower() for c in cab.columns)
                    if 12 <= len(cab.columns) <= 20 and 'estaci' in cabeceras_l:
                        enc_path = ruta
                except Exception:
                    pass
        elif nombre_l.endswith('.csv'):
            if 'infonieve' in nombre_l:
                inf_path = ruta
            elif inf_path is None:
                try:
                    cab = pd.read_csv(ruta, nrows=0, encoding='latin-1', sep=None, engine='python')
                    if 'tipo_nieve' in cab.columns or 'espesor_minimo' in cab.columns:
                        inf_path = ruta
                except Exception:
                    pass
    return enc_path, inf_path

print('Detectando archivos en', CONTENT, '...')
ruta_enc, ruta_inf = detectar_archivos(CONTENT)

if ruta_enc is None:
    raise FileNotFoundError(f'No se encontró ningún .xlsx válido en {CONTENT}')
if ruta_inf is None:
    raise FileNotFoundError(f'No se encontró el CSV de Infonieve en {CONTENT}')

print(f'  Encuesta  : {os.path.basename(ruta_enc)}')
print(f'  Infonieve : {os.path.basename(ruta_inf)}')


Detectando archivos en /content/ ...
  Encuesta  : Tu experiencia en las estaciones de esquí (respuestas) (3).xlsx
  Infonieve : infonieve-partedenieve-historico-estaciones.csv


---
## Parte 1 — Encuesta
### 1.1 Carga y renombrado de columnas

In [ ]:
enc_raw = pd.read_excel(ruta_enc)
enc = enc_raw.copy()
print(f'Filas brutas: {len(enc)}')

enc.columns = [
    'marca_temporal',        # 0
    'edad_rango',            # 1
    'ciudad',                # 2
    'dias_temporada',        # 3
    'estaciones_habituales', # 4
    'dificultad_acceso',     # 5
    'hora_llegada',          # 6
    'colas',                 # 7
    'tiempo_colas',          # 8
    'valoracion',            # 9
    'mejoras_sugeridas',     # 10
    'mejor_estacion',        # 11
    'utilidad_app',          # 12
    'features_app',          # 13
]
enc = enc.drop(columns=['marca_temporal'])
enc = enc.dropna(subset=['valoracion'])
enc['valoracion'] = pd.to_numeric(enc['valoracion'], errors='coerce').astype('Int64')

for col in ['ciudad', 'mejoras_sugeridas', 'features_app']:
    enc[col] = enc[col].apply(limpiar_texto_libre)

print(f'Filas tras filtrar sin valoración: {len(enc)}')
print(f'Columnas: {list(enc.columns)}')


Filas brutas: 224
Filas tras filtrar sin valoración: 224
Columnas: ['edad_rango', 'ciudad', 'dias_temporada', 'estaciones_habituales', 'dificultad_acceso', 'hora_llegada', 'colas', 'tiempo_colas', 'valoracion', 'mejoras_sugeridas', 'mejor_estacion', 'utilidad_app', 'features_app']


### 1.2 Normalización: mejor_estacion

In [ ]:
MEJOR_EST_MAP = {
    'baqueira/beret':'Baqueira/Beret','baqueira/beret/bonaigua':'Baqueira/Beret',
    'baqueira beret':'Baqueira/Beret','baqueiraberet':'Baqueira/Beret',
    'baqueira-beret':'Baqueira/Beret','baqueira':'Baqueira/Beret','beret':'Baqueira/Beret',
    'siempre voy a formigal en mis 40 anos solo he esquiado en formigal':'Formigal/Panticosa',
    'formigal/panticosa':'Formigal/Panticosa','formigal':'Formigal/Panticosa',
    'firmigal':'Formigal/Panticosa','solo esquio en formigal':'Formigal/Panticosa',
    'astun/candanchu':'Astún/Candanchú','astun candanchu':'Astún/Candanchú',
    'astun':'Astún/Candanchú','candanchu':'Astún/Candanchú',
    'sierra nevada':'Sierra Nevada','sierra mevada':'Sierra Nevada',
    'valdesqui':'Valdesquí',
    'la molina / masella':'La Molina/Masella','la molina/ masella':'La Molina/Masella',
    'la molina masella':'La Molina/Masella','la molina':'La Molina/Masella','masella':'La Molina/Masella',
    'cerler':'Cerler','javalambre':'Javalambre','san isidro':'San Isidro',
    'llanos del hospital':'Llanos del Hospital','port aine':'Port Ainé','portaine':'Port Ainé',
    'aramon':'Aramón',
    'grandvalira':'Extranjero','grand valira':'Extranjero','granvalira':'Extranjero',
    'grandvarila':'Extranjero','grandvallira':'Extranjero','andorra':'Extranjero',
    'andorra y los alpes':'Extranjero','alpes mucha gente pero tan grande que no hay problemas de colas':'Extranjero',
    'cualquiera de los alpes':'Extranjero','cualquiera fuera de espana':'Extranjero',
    'bien gestionada solo en el extranjero':'Extranjero','las del extranjero':'Extranjero',
    'las de colorado usa':'Extranjero','corchevel':'Extranjero','courchevel en francia':'Extranjero',
    'megeve':'Extranjero','avoriaz':'Extranjero','dolomitas val gardena':'Extranjero',
    'pas de la casa':'Extranjero','el tarter':'Extranjero','peyragudes':'Extranjero',
    'piau engaly':'Extranjero','port du soleil':'Extranjero','los 3 valles':'Extranjero',
    'las estaciones austriacas':'Extranjero','pirineo frances':'Extranjero',
    'pirineo frances y alpes':'Extranjero',
    'yo voy a italia porque en espana es caro se aparca mal y hay colas':'Extranjero',
    'de las que he esquiado recientemente portes du soleil francia pero normalmente voy a sierra nevada que es un poco caotica':'Extranjero',
    'grandvalira gestion de parking cues remontadors si tienes que esperar mas de 3 minutos ya encuentras que hay demasiada gente':'Extranjero',
    'ninguna':'Ninguna','ninguna destaca':'Ninguna','de las que voy ninguna':'Ninguna',
    'todas /  igual de deficiente':'Ninguna','todas  igual de deficiente':'Ninguna',
    'en espana ninguna':'Ninguna','en espana pocas':'Ninguna',
    'es un problema generalizado los festivos y fines de semana':None,
    'la fila de uno en los remontes para rellenar huecos':None,
    'no lo se':'No lo sé','no se':'No lo sé','no sabria decir':'No lo sé',
    'no tengo criterio':'No lo sé','no tengo informacion en espana':'No lo sé',
    'no te puedo decir ahora':'No lo sé','desconozco':'No lo sé',
    'ns/nc':None,'n/d':None,'ns':None,'nd':None,
    'a las que voy bastante similar':None,
    'port aine esta bien gestionada pero tambien porque quiza es demasiado pequena':'Port Ainé',
}

_PATRON_EXTRANJERO = re.compile(
    r'alpes|andorra|grandvalira|granvalira|grand valira|grandvallira|grandvarila'
    r'|fuera de espa|dolomita|austria|suiza|italia|francia|colorado'
    r'|piau|peyragudes|avoriaz|courchevel|corchevel|megeve|val gardena'
    r'|port du soleil|portes du soleil|pas de la casa|el tarter|extranjer'
    r'|europeas|3 valles|tres valles|austriacas|palisades|aosta|pirineo franc',
    re.IGNORECASE
)
_PATRON_NO_SE = re.compile(
    r'^(no\s+(lo\s+)?se[ñn]?[e]?|ns[/\s]?nc|n[/\s]?d|desconozco|no sabr|no tengo|no te puedo|no sab)',
    re.IGNORECASE
)
_PATRON_NINGUNA = re.compile(r'^(ninguna|todas.+(igual|deficiente)|en espa.+ninguna|pocas)', re.IGNORECASE)

def normalizar_mejor_estacion(texto):
    if pd.isna(texto) or not isinstance(texto, str): return np.nan
    t_limpio = quitar_emoji(texto).strip()
    if not t_limpio or re.fullmatch(r'[.\-–—/\s]+', t_limpio): return np.nan
    clave = normalizar_clave(t_limpio)
    if clave in MEJOR_EST_MAP: return MEJOR_EST_MAP[clave]
    if _PATRON_EXTRANJERO.search(t_limpio): return 'Extranjero'
    if _PATRON_NO_SE.search(t_limpio): return 'No lo sé'
    if _PATRON_NINGUNA.search(t_limpio): return 'Ninguna'
    if len(t_limpio) > 40: return np.nan
    return np.nan

def limpiar_mejor_estacion(texto):
    if pd.isna(texto): return np.nan
    t = quitar_emoji(str(texto)).strip()
    if re.fullmatch(r'[.\-–—/\s]+', t): return np.nan
    return t if t else np.nan

enc['mejor_estacion_limpia'] = enc['mejor_estacion'].apply(limpiar_mejor_estacion)
enc['mejor_estacion_norm']   = enc['mejor_estacion'].apply(normalizar_mejor_estacion)
print(f"mejor_estacion_norm — valores únicos: {enc['mejor_estacion_norm'].value_counts().to_dict()}")


mejor_estacion_norm — valores únicos: {'Baqueira/Beret': 79, 'Extranjero': 37, 'Formigal/Panticosa': 24, 'Ninguna': 20, 'No lo sé': 15, 'Sierra Nevada': 9, 'Cerler': 8, 'Astún/Candanchú': 7, 'La Molina/Masella': 5, 'Valdesquí': 1, 'Aramón': 1, 'Port Ainé': 1, 'Llanos del Hospital': 1, 'San Isidro': 1, 'Javalambre': 1}


### 1.3 Normalización: estaciones_habituales + variables binarias

In [ ]:
EST_HAB_MAP = {
    'baqueira/beret':'Baqueira/Beret','baqueira/beret/bonaigua':'Baqueira/Beret',
    'baqueira beret':'Baqueira/Beret','baqueira-beret':'Baqueira/Beret',
    'baqueiraberet':'Baqueira/Beret','baqueira':'Baqueira/Beret','beret':'Baqueira/Beret',
    'formigal/panticosa':'Formigal/Panticosa','formigal':'Formigal/Panticosa',
    'firmigal':'Formigal/Panticosa','panticosa':'Formigal/Panticosa',
    'astun/candanchu':'Astún/Candanchú','astun candanchu':'Astún/Candanchú',
    'astun':'Astún/Candanchú','candanchu':'Astún/Candanchú',
    'sierra nevada':'Sierra Nevada','valdesqui':'Valdesquí',
    'la molina / masella':'La Molina/Masella','la molina/ masella':'La Molina/Masella',
    'la molina masella':'La Molina/Masella','la molina':'La Molina/Masella','masella':'La Molina/Masella',
    'cerler':'Cerler','port aine':'Port Ainé','portaine':'Port Ainé',
    'port aine o grandvalira':'Port Ainé',
    'san isidro':'San Isidro',
    'llanos del hospital y navafria':'Llanos del Hospital','llanos del hospital':'Llanos del Hospital',
    'navafria':'Navafría','ezcaray':'Ezcaray','xanadu':'Xanadú','snozone':'Xanadú',
    'alto campoo':'Alto Campoo','alto campo':'Alto Campoo','javalambre':'Javalambre',
    'la pinilla':'La Pinilla','valdelinares':'Valdelinares',
    'puerto de navacerrada':'Puerto de Navacerrada','aramon':'Aramón',
    'grandvalira':'Extranjero','grand valira':'Extranjero','granvalira':'Extranjero',
    'grandvarila':'Extranjero','grandvallira':'Extranjero','andorra':'Extranjero',
    'andorra y alpes':'Extranjero','alpes':'Extranjero','austria':'Extranjero',
    'aosta':'Extranjero','suiza':'Extranjero','dolomitas':'Extranjero','piau':'Extranjero',
    'peyragudes':'Extranjero','port du soleil':'Extranjero','pas de la casa':'Extranjero',
    'palisades tahoe':'Extranjero','pirineo frances y alpes':'Extranjero',
    'pirineo frances':'Extranjero','las tres valles':'Extranjero','los 3 valles':'Extranjero',
    'estaciones europeas':'Extranjero','fuera de espana':'Extranjero','extranjero':'Extranjero',
    'en los alpes':'Extranjero','varias de andorra':'Extranjero',
    '':None,' ':None,
}
_COMPUESTOS_NO_PARTIR = {'llanos del hospital y navafria','andorra y los alpes'}

def _token_a_canonica(token):
    t = token.strip()
    if not t: return None
    clave = normalizar_clave(t)
    if clave in EST_HAB_MAP: return EST_HAB_MAP[clave]
    if _PATRON_EXTRANJERO.search(t): return 'Extranjero'
    return None

def procesar_estaciones_habituales(valor):
    if pd.isna(valor) or str(valor).strip() == '': return []
    resultado = []
    segmentos = [s.strip() for s in str(valor).split(',')]
    for seg in segmentos:
        if not seg: continue
        clave_seg = normalizar_clave(seg)
        if clave_seg in _COMPUESTOS_NO_PARTIR or clave_seg in EST_HAB_MAP:
            canon = _token_a_canonica(seg)
            if canon: resultado.append(canon)
            continue
        sub_tokens = re.split(r'\s+[yo]\s+', seg, flags=re.IGNORECASE)
        for tok in sub_tokens:
            canon = _token_a_canonica(tok)
            if canon: resultado.append(canon)
    vistos, unique = set(), []
    for x in resultado:
        if x not in vistos: vistos.add(x); unique.append(x)
    return unique

enc['estaciones_habituales_limpia'] = enc['estaciones_habituales'].apply(
    lambda v: quitar_emoji(str(v)).strip().rstrip(', ') if not pd.isna(v) else np.nan)
enc['estaciones_habituales_norm'] = enc['estaciones_habituales'].apply(
    lambda v: ', '.join(procesar_estaciones_habituales(v)) or np.nan)

ESTACIONES_BINARIAS = {
    'Baqueira/Beret':'hab_baqueira_beret','Formigal/Panticosa':'hab_formigal_panticosa',
    'Astún/Candanchú':'hab_astun_candanchu','Sierra Nevada':'hab_sierra_nevada',
    'Valdesquí':'hab_valdesqui','La Molina/Masella':'hab_la_molina_masella',
    'Cerler':'hab_cerler','Port Ainé':'hab_port_aine','San Isidro':'hab_san_isidro',
    'Llanos del Hospital':'hab_llanos_hospital','Navafría':'hab_navafria',
    'Ezcaray':'hab_ezcaray','Xanadú':'hab_xanadu','Alto Campoo':'hab_alto_campoo',
    'Javalambre':'hab_javalambre','La Pinilla':'hab_la_pinilla',
    'Valdelinares':'hab_valdelinares','Puerto de Navacerrada':'hab_puerto_navacerrada',
    'Aramón':'hab_aramon','Extranjero':'hab_extranjero',
}
for est, col_bin in ESTACIONES_BINARIAS.items():
    enc[col_bin] = enc['estaciones_habituales_norm'].apply(
        lambda x, e=est: 1 if isinstance(x, str) and e in x else 0)

print(f'estaciones_habituales_norm — shape encuesta: {enc.shape}')
print('Variables binarias hab_*:', [c for c in enc.columns if c.startswith('hab_')])


estaciones_habituales_norm — shape encuesta: (224, 37)
Variables binarias hab_*: ['hab_baqueira_beret', 'hab_formigal_panticosa', 'hab_astun_candanchu', 'hab_sierra_nevada', 'hab_valdesqui', 'hab_la_molina_masella', 'hab_cerler', 'hab_port_aine', 'hab_san_isidro', 'hab_llanos_hospital', 'hab_navafria', 'hab_ezcaray', 'hab_xanadu', 'hab_alto_campoo', 'hab_javalambre', 'hab_la_pinilla', 'hab_valdelinares', 'hab_puerto_navacerrada', 'hab_aramon', 'hab_extranjero']


### 1.4 Codificaciones ordinales (_escala)

In [ ]:
enc['edad_escala'] = enc['edad_rango'].map({
    'Menos de 18 años':1,'18-30 años':2,'31-45 años':3,'46-60 años':4,'Más de 60 años':5})

MAPA_DIAS_LIMPIO = {'1-3 días':'1-3 días','4-7 días':'4-7 días','8-15 días':'8-15 días',
                    '16-24 días':'16+ días','Más de 20 días':'16+ días','Más de 25 días':'16+ días'}
enc['dias_temporada_limpia'] = enc['dias_temporada'].map(MAPA_DIAS_LIMPIO)
enc['dias_escala'] = enc['dias_temporada_limpia'].map({'1-3 días':1,'4-7 días':2,'8-15 días':3,'16+ días':4})

enc['hora_llegada_escala'] = enc['hora_llegada'].map({
    'Antes de las 8:00':1,'Entre las 8:00 y las 9:00':2,
    'Entre las 9:00 y las 10:00':3,'Después de las 10:00':4})

enc['dificultad_acceso_escala'] = enc['dificultad_acceso'].map(
    {'Nunca':0,'A veces':1,'Frecuentemente':2,'Siempre':3})

enc['colas_escala'] = enc['colas'].map({'Nunca':0,'A veces':1,'Frecuentemente':2,'Siempre':3})

enc['tiempo_colas_escala'] = enc['tiempo_colas'].map(
    {'Menos de 10 min':1,'10-30 min':2,'30-60 min':3,'Más de 1 hora':4})

enc['utilidad_app_escala'] = enc['utilidad_app'].map(
    {'No':0,'No lo sé':1,'Sí, algo':2,'Sí, mucho':3})

print('Codificaciones _escala añadidas.')
print('Shape encuesta:', enc.shape)


Codificaciones _escala añadidas.
Shape encuesta: (224, 45)


### 1.5 Exportar encuesta limpia

In [ ]:
enc.to_csv(SALIDA_ENC, index=False, encoding='utf-8-sig')
print(f'✓ Guardado: {SALIDA_ENC}')
print(f'  Shape: {enc.shape}')
print(f'  Columnas: {list(enc.columns)}')


✓ Guardado: /content/encuesta_clean_tfg.csv
  Shape: (224, 45)
  Columnas: ['edad_rango', 'ciudad', 'dias_temporada', 'estaciones_habituales', 'dificultad_acceso', 'hora_llegada', 'colas', 'tiempo_colas', 'valoracion', 'mejoras_sugeridas', 'mejor_estacion', 'utilidad_app', 'features_app', 'mejor_estacion_limpia', 'mejor_estacion_norm', 'estaciones_habituales_limpia', 'estaciones_habituales_norm', 'hab_baqueira_beret', 'hab_formigal_panticosa', 'hab_astun_candanchu', 'hab_sierra_nevada', 'hab_valdesqui', 'hab_la_molina_masella', 'hab_cerler', 'hab_port_aine', 'hab_san_isidro', 'hab_llanos_hospital', 'hab_navafria', 'hab_ezcaray', 'hab_xanadu', 'hab_alto_campoo', 'hab_javalambre', 'hab_la_pinilla', 'hab_valdelinares', 'hab_puerto_navacerrada', 'hab_aramon', 'hab_extranjero', 'edad_escala', 'dias_temporada_limpia', 'dias_escala', 'hora_llegada_escala', 'dificultad_acceso_escala', 'colas_escala', 'tiempo_colas_escala', 'utilidad_app_escala']


---
## Parte 2 — Infonieve
### 2.1 Carga y limpieza básica

In [ ]:
inf_raw = pd.read_csv(ruta_inf, sep=None, engine='python', encoding='latin-1')
inf = inf_raw.copy()
print(f'Filas brutas: {len(inf):,}  ·  Columnas: {inf.shape[1]}')
print('Columnas:', list(inf.columns))

# Reemplazar '-' por NaN en columnas operativas
COLS_CON_DASH = ['remontes_abiertos','pistas_abiertas','kilometros_abiertos',
                 'espesor_minimo','espesor_maximo','tipo_nieve']
for col in COLS_CON_DASH:
    inf[col] = inf[col].replace('-', np.nan)

# Convertir numéricos con coma decimal
COLS_COMA = ['kilometros_total','kilometros_abiertos','espesor_minimo','espesor_maximo']
for col in COLS_COMA:
    inf[col] = pd.to_numeric(
        inf[col].astype(str).str.replace(',','.', regex=False)
                             .replace({'nan':np.nan,'None':np.nan}), errors='coerce')

for col in ['remontes_abiertos','pistas_abiertas']:
    inf[col] = pd.to_numeric(inf[col], errors='coerce').astype('Int64')

print('\nNulos tras limpieza básica:')
print(inf[COLS_CON_DASH].isnull().sum())


Filas brutas: 142,893  ·  Columnas: 12
Columnas: ['estacion', 'fecha', 'estado_abierta', 'remontes_abiertos', 'remontes_total', 'pistas_abiertas', 'pistas_total', 'kilometros_abiertos', 'kilometros_total', 'espesor_minimo', 'espesor_maximo', 'tipo_nieve']

Nulos tras limpieza básica:
remontes_abiertos      98419
pistas_abiertas        98616
kilometros_abiertos    98937
espesor_minimo         98281
espesor_maximo         97277
tipo_nieve             95963
dtype: int64


### 2.2 Variables temporales y temporada de esquí

In [ ]:
inf['fecha']      = pd.to_datetime(inf['fecha'], errors='coerce')
inf['anio']       = inf['fecha'].dt.year
inf['mes']        = inf['fecha'].dt.month
inf['dia_semana'] = inf['fecha'].dt.dayofweek
inf['es_finde']   = (inf['dia_semana'] >= 5).astype(int)
inf['nombre_mes'] = inf['mes'].map({1:'enero',2:'febrero',3:'marzo',4:'abril',5:'mayo',
    6:'junio',7:'julio',8:'agosto',9:'septiembre',10:'octubre',11:'noviembre',12:'diciembre'})

def calcular_temporada(row):
    mes, anio = row['mes'], row['anio']
    if pd.isna(mes) or pd.isna(anio):
        return pd.Series({'es_temporada_esqui':False,'temporada_esqui':np.nan,'temporada_inicio':np.nan})
    mes, anio = int(mes), int(anio)
    if mes in {10,11,12}:
        return pd.Series({'es_temporada_esqui':True,'temporada_esqui':f'{anio}/{anio+1}','temporada_inicio':anio})
    elif mes in {1,2,3,4,5}:
        return pd.Series({'es_temporada_esqui':True,'temporada_esqui':f'{anio-1}/{anio}','temporada_inicio':anio-1})
    else:
        return pd.Series({'es_temporada_esqui':False,'temporada_esqui':np.nan,'temporada_inicio':np.nan})

temp_cols = inf[['mes','anio']].apply(calcular_temporada, axis=1)
inf = pd.concat([inf, temp_cols], axis=1)
inf['temporada_inicio'] = pd.to_numeric(inf['temporada_inicio'], errors='coerce').astype('Int64')
print('Variables temporales añadidas.')


Variables temporales añadidas.


### 2.3 Limpieza de tipo_nieve y variables binarias

In [ ]:
ESTADOS_OPERATIVOS = {'Cerrada','Previsión Apertura','Abierta Fin de Semana','Uso turístico','Artificial'}
TYPOS_NIEVE = {
    'Polvo/dura':'Polvo/Dura','Polvo/Pisada':'Polvo/Dura','Diura':'Dura',
    '-húmeda':'Húmeda','pol':'Polvo','?':np.nan,'.':np.nan,
    'Húmeda/Dura':'Dura/Húmeda','Primavera/Dura':'Dura/Primavera',
    'Primavera/Húmeda':'Húmeda/Primavera',
}
inf['tipo_nieve_limpia'] = (
    inf['tipo_nieve'].replace(TYPOS_NIEVE)
    .apply(lambda x: np.nan if x in ESTADOS_OPERATIVOS else x)
)

def contiene_tipo(serie, patron):
    return serie.apply(lambda x: 1 if isinstance(x,str) and bool(re.search(patron,x,re.IGNORECASE)) else 0)

inf['nieve_polvo']      = contiene_tipo(inf['tipo_nieve_limpia'], r'polvo')
inf['nieve_dura']       = contiene_tipo(inf['tipo_nieve_limpia'], r'dura')
inf['nieve_humeda']     = contiene_tipo(inf['tipo_nieve_limpia'], r'h[úu]meda')
inf['nieve_primavera']  = contiene_tipo(inf['tipo_nieve_limpia'], r'primavera')
inf['nieve_artificial'] = contiene_tipo(inf['tipo_nieve_limpia'], r'artificial')
inf['nieve_pisada']     = contiene_tipo(inf['tipo_nieve_limpia'], r'pisada')

print('tipo_nieve_limpia — valores únicos:')
print(inf['tipo_nieve_limpia'].value_counts(dropna=False).head(15))


tipo_nieve_limpia — valores únicos:
tipo_nieve_limpia
NaN                     97535
Polvo                   16158
Polvo/Dura              11233
Primavera                5883
Húmeda                   3735
Dura                     3628
Dura/Primavera           2854
Polvo/Primavera           707
Polvo/Húmeda              650
Dura/Húmeda               274
Húmeda/Primavera          107
Polvo/Artificial           48
Dura/Artificial            46
Húmeda/Artificial          13
Polvo/Dura/Primavera       12
Name: count, dtype: int64


### 2.4 Ratios de apertura

In [ ]:
inf['pct_remontes_abiertos'] = np.where(
    inf['remontes_total'] > 0, inf['remontes_abiertos'] / inf['remontes_total'], np.nan)
inf['pct_pistas_abiertas'] = np.where(
    inf['pistas_total'] > 0, inf['pistas_abiertas'] / inf['pistas_total'], np.nan)
inf['pct_km_abiertos'] = np.where(
    inf['kilometros_total'] > 0, inf['kilometros_abiertos'] / inf['kilometros_total'], np.nan)

print('Ratios calculados.')
print('Shape infonieve:', inf.shape)


Ratios calculados.
Shape infonieve: (142893, 30)


### 2.5 Exportar infonieve limpio

In [ ]:
inf.to_csv(SALIDA_INF, index=False, encoding='utf-8-sig')
print(f'✓ Guardado: {SALIDA_INF}')
print(f'  Shape: {inf.shape}')


✓ Guardado: /content/infonieve_clean_tfg.csv
  Shape: (142893, 30)


---
## Corrección final — mejor_estacion_norm (v3)
Aplica sobre el CSV ya guardado para asignar 'No lo sé' a respuestas ambiguas (emojis de duda, ns/nc, etc.)

In [ ]:
_TOKENS_NO_SE = {'ns','ns/nc','n/d','nd'}
_PATRONES_DUDA = [
    r'^n[o]?\s*s[e]?$', r'^no\s+lo\s+s[e]?$', r'^ns[/]?nc$', r'^n[/]?d$',
    r'^desconozco$', r'^no\s+sabr', r'^no\s+tengo\s+(criterio|informaci)',
    r'^no\s+te\s+puedo\s+decir',
]

def _es_no_se(texto):
    if not isinstance(texto, str) or not texto.strip(): return np.nan
    clave = normalizar_clave(texto)
    if clave in _TOKENS_NO_SE: return 'No lo sé'
    sin_emoji = _RE_EMOJI.sub('', texto).strip()
    sin_emoji_norm = normalizar_clave(sin_emoji)
    if not sin_emoji_norm and _RE_EMOJI.search(texto): return 'No lo sé'
    for pat in _PATRONES_DUDA:
        if re.match(pat, sin_emoji_norm): return 'No lo sé'
    return np.nan

df_enc = pd.read_csv(SALIDA_ENC, low_memory=False)
df_enc['mejor_estacion_limpia'] = df_enc['mejor_estacion_limpia'].apply(quitar_emoji)
mask_norm_nan = df_enc['mejor_estacion_norm'].isna()
df_enc.loc[mask_norm_nan, 'mejor_estacion_norm'] = df_enc.loc[mask_norm_nan, 'mejor_estacion'].apply(_es_no_se)
df_enc.to_csv(SALIDA_ENC, index=False)

print(f'Shape final encuesta: {df_enc.shape}')
print(f'NaN en mejor_estacion_norm: {df_enc["mejor_estacion_norm"].isna().sum()}')
print('✓ Corrección final aplicada y guardada.')


Shape final encuesta: (224, 45)
NaN en mejor_estacion_norm: 10
✓ Corrección final aplicada y guardada.


---
## Validación final

In [ ]:
enc_v  = pd.read_csv(SALIDA_ENC)
inf_v  = pd.read_csv(SALIDA_INF)

print('=' * 55)
print(f'encuesta_clean_tfg.csv  : {enc_v.shape[0]} filas × {enc_v.shape[1]} columnas')
print(f'infonieve_clean_tfg.csv : {inf_v.shape[0]:,} filas × {inf_v.shape[1]} columnas')
print()
print('Nulos encuesta (sólo cols con nulos):')
nu = enc_v.isnull().sum()
for c,n in nu[nu>0].items():
    print(f'  {c}: {n} ({n/len(enc_v)*100:.1f}%)')
print()
print('Nulos infonieve (sólo cols con nulos):')
ni = inf_v.isnull().sum()
for c,n in ni[ni>0].items():
    print(f'  {c}: {n} ({n/len(inf_v)*100:.1f}%)')
print('=' * 55)
print('✅ Limpieza completada sin errores.')


encuesta_clean_tfg.csv  : 224 filas × 45 columnas
infonieve_clean_tfg.csv : 142,893 filas × 30 columnas

Nulos encuesta (sólo cols con nulos):
  features_app: 71 (31.7%)
  mejor_estacion_limpia: 5 (2.2%)
  mejor_estacion_norm: 10 (4.5%)

Nulos infonieve (sólo cols con nulos):
  remontes_abiertos: 98419 (68.9%)
  pistas_abiertas: 98616 (69.0%)
  kilometros_abiertos: 98937 (69.2%)
  espesor_minimo: 98281 (68.8%)
  espesor_maximo: 97277 (68.1%)
  tipo_nieve: 95963 (67.2%)
  temporada_esqui: 42087 (29.5%)
  temporada_inicio: 42087 (29.5%)
  tipo_nieve_limpia: 97535 (68.3%)
  pct_remontes_abiertos: 98419 (68.9%)
  pct_pistas_abiertas: 98616 (69.0%)
  pct_km_abiertos: 98937 (69.2%)
✅ Limpieza completada sin errores.


In [ ]:
from google.colab import files

files.download('/content/encuesta_clean_tfg.csv')
files.download('/content/infonieve_clean_tfg.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>